# 🏌️ DOH 스윙분석 — 원클릭 (Colab 무료)

**하는 법:** 위에서 **런타임 ▸ 런타임 유형 변경 ▸ T4 GPU ▸ 저장**, 그다음 **런타임 ▸ 모두 실행**.
맨 아래에 화면이 뜨면 거기서 **영상 올리고 [분석하기]** 만 누르면 됨. (셀 하나씩 안 만져도 됨)

### 1칸 — 준비 (모델 로드)
`>>> NLF OK` 뜨면 성공. 처음 한 번만 오래 걸림.

In [ ]:
# 1칸 · 준비 (1~2분). 끝에 >>> NLF OK 뜨면 성공.
!pip -q install "gradio>=4" 2>/dev/null
import torch, torchvision, torchvision.ops, os, urllib.request
print("torch", torch.__version__, "| GPU", torch.cuda.is_available())
URL = "https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript"
M = "nlf_l_multi_0.3.2.torchscript"
if (not os.path.exists(M)) or os.path.getsize(M) < 10_000_000:
    print("NLF 모델 다운로드(470MB)..."); urllib.request.urlretrieve(URL, M)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.jit.load(M).to(DEVICE).eval()
print(">>> NLF OK  (device =", DEVICE, ")")

### 2칸 — 분석 화면
실행하면 **바로 아래에 업로드 화면**이 떠. 영상 올리고 각도(정면/측면)·주손 고른 뒤 **분석하기**.

In [ ]:
# 2칸 · UI 실행. 실행하면 이 아래에 화면이 뜸 → 영상 올리고 [분석하기].
BR = "claude/session-context-recovery-qrdpov"
BASE = f"https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/{BR}/pose3d_poc"
import urllib.request, numpy as np, cv2, gradio as gr
for f in ("wham_golf_rotation.py", "wham_golf_metrics.py"):
    urllib.request.urlretrieve(f"{BASE}/{f}", f)
from wham_golf_rotation import build_v1

def video_to_joints(path, max_side=720):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 60.0; J = []
    with torch.inference_mode():
        while True:
            ok, fr = cap.read()
            if not ok: break
            h, w = fr.shape[:2]; sc = max_side / max(h, w)
            if sc < 1: fr = cv2.resize(fr, (int(w*sc), int(h*sc)), interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            pred = model.detect_smpl_batched(t); per = pred["joints3d"][0]
            if per is None or len(per) == 0: continue
            kp = per[0]; kp = kp.detach().cpu().numpy() if torch.is_tensor(kp) else np.asarray(kp)
            J.append(kp)
    cap.release(); return np.asarray(J), round(float(fps), 2)

# ── 표시 어휘 (analyzer2 v21 팔레트/구조에 맞춤) ──
META = {
 "VF015":("어깨 회전 @탑","회전"),"VF018":("골반 회전 @탑","회전"),"VF020":("X-Factor @탑","회전"),"VF075":("힙 클리어","회전"),
 "VF002":("척추각 @어드레스","자세 (측면)"),"VF038":("자세 유지 (어드→탑)","자세 (측면)"),
 "VF076":("척추각 유지","자세 (측면)"),"VF022":("어깨 플레인","자세 (측면)"),"VF001":("좌우 틸트","자세 (정면)"),
 "VF011":("리드팔 곧음","팔·무릎"),"VF012":("트레일팔 굽힘","팔·무릎"),"VF027":("트레일 팔꿈치각","팔·무릎"),
 "VF087":("리드팔 굽힘 @임팩트","팔·무릎"),"VF039":("리드무릎 굽힘변화","팔·무릎"),
 "VF040":("트레일무릎 굽힘","팔·무릎"),"VF088":("리드무릎각 @임팩트","팔·무릎"),
 "VF031":("머리 스웨이","스웨이 (정면)"),"VF034":("골반 스웨이","스웨이 (정면)"),
 "VF113":("백스윙 시간","템포"),"VF114":("다운스윙 시간","템포"),"VF111":("템포 비율","템포"),
}
FLAG = {"view_mismatch":"이 각도에선 측정 불가","off_axis_view":"각도 미상·참고용","depth_unreliable":"깊이 불안정",
        "interpolated_event":"이벤트 보간","club_not_detected":"클럽 미검출"}
GROUPS = ["자세 (측면)","자세 (정면)","팔·무릎","스웨이 (정면)","템포"]   # 회전은 전용 섹션에서
# P구간 라벨(한글) + 순서. P2/P6/P8=샤프트 이벤트(클럽 검출 필요 → pose만으론 미검출)
PLBL = {"P1":"어드레스","P2":"토업(샤프트평행)","P3":"리드팔평행(BS)","P4":"탑","P5":"리드팔평행(DS)",
        "P6":"샤프트평행(DS)","P7":"임팩트","P8":"샤프트평행(FT)","P9":"리드팔평행(FT)","P10":"피니시"}
PORD = ["P1","P2","P3","P4","P5","P6","P7","P8","P9","P10"]

# analyzer2 v21 팔레트
CSS = """<style>
.doh *{box-sizing:border-box}
.doh{font:14px/1.5 -apple-system,'Segoe UI','Malgun Gothic',sans-serif;background:#0f1216;
 border:1px solid #262d38;border-radius:12px;padding:16px 18px;color:#e8edf2}
.doh h2{font-size:12px;text-transform:uppercase;letter-spacing:.5px;color:#8b97a5;margin:18px 0 9px}
.doh .meta{display:flex;gap:8px;flex-wrap:wrap}
.doh .chip{background:#1a1f27;border:1px solid #33404f;border-radius:9px;padding:6px 11px;font-size:12px;color:#8b97a5}
.doh .chip b{color:#4ea1ff;font-variant-numeric:tabular-nums}
.doh .evrow{display:flex;flex-wrap:wrap;gap:6px}
.doh .ev{background:#1a1f27;border:1px solid #33404f;border-radius:8px;padding:5px 9px;font-size:12px}
.doh .ev b{color:#ff7eb6}
.doh .ev.miss{opacity:.4}
.doh .ev.itp b{color:#ffb648}
.doh .ev .fr{color:#8b97a5;font-variant-numeric:tabular-nums}
.doh .mgrid{display:grid;grid-template-columns:2fr 1fr;gap:10px}
@media(max-width:560px){.doh .mgrid{grid-template-columns:1fr}}
.doh .mcard{background:#161b22;border:1px solid #33404f;border-radius:10px;padding:12px 14px}
.doh .mcard h4{margin:0 0 8px;font-size:12px;color:#4ea1ff;letter-spacing:.3px;font-weight:700}
.doh .mrow{display:flex;justify-content:space-between;align-items:baseline;padding:5px 0;border-bottom:1px solid #222b35}
.doh .mrow:last-child{border-bottom:none}
.doh .mrow .lab{color:#8b97a5;font-size:12px}
.doh .mrow .val{font-size:22px;font-weight:800;font-variant-numeric:tabular-nums;color:#e8edf2}
.doh .mrow .val.xf{color:#ffb648}
.doh .mrow .val.na{font-size:13px;font-weight:700;color:#6e7681}
.doh .grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:10px}
.doh .fc{background:#161b22;border:1px solid #33404f;border-radius:10px;padding:12px}
.doh .fc .nm{font-size:12px;color:#8b97a5;min-height:30px;line-height:1.3}
.doh .fc .fv{font-size:24px;font-weight:800;color:#e8edf2;font-variant-numeric:tabular-nums}
.doh .fc .fv .u{font-size:12px;color:#8b97a5;font-weight:600}
.doh .fc .na{font-size:14px;font-weight:700;color:#6e7681}
.doh .bar{margin-top:8px;height:4px;border-radius:2px;background:#222b35}
.doh .bar>span{display:block;height:100%;border-radius:2px;background:#3ecf8e}
.doh .flag{margin-top:6px;font-size:11px;color:#ffb648}
.doh .foot{margin-top:16px;font-size:13px;color:#8b97a5}
.doh .foot b{color:#e8edf2}
.doh .note{color:#8b97a5;font-size:11px;margin-top:8px;line-height:1.55}
</style>"""

def _num(f):
    """(값문자열|None, 신뢰도%, 플래그텍스트). None이면 측정불가."""
    flags = " · ".join(FLAG.get(x, x) for x in f.get("error_flags", []))
    if f.get("value") is None:
        return None, 0, (flags or "측정 불가")
    u = "°" if f["unit"]=="deg" else ("" if f["unit"]=="ratio" else " "+f["unit"])
    return f'{f["value"]}<span class="u">{u}</span>', int(round(f.get("confidence",0)*100)), flags

def _fc(f):
    name = META.get(f["feature_id"], (f.get("name",""),"기타"))[0]
    v, cf, flags = _num(f)
    if v is None:
        body = f'<div class="na">{flags}</div>'
    else:
        body = f'<div class="fv">{v}</div><div class="bar"><span style="width:{cf}%"></span></div>'
        if flags: body += f'<div class="flag">{flags}</div>'
    return f'<div class="fc"><div class="nm">{name}</div>{body}</div>'

def _mrow(lab, f, cls=""):
    if f is None:
        return f'<div class="mrow"><span class="lab">{lab}</span><span class="val na">—</span></div>'
    v, cf, flags = _num(f)
    if v is None:
        return f'<div class="mrow"><span class="lab">{lab}</span><span class="val na">{flags or "측정불가"}</span></div>'
    return f'<div class="mrow"><span class="lab">{lab}</span><span class="val {cls}">{v}</span></div>'

def render(inst):
    ev  = {e["p"]: e["frame"]  for e in inst.get("swing_events", [])}
    evm = {e["p"]: e["method"] for e in inst.get("swing_events", [])}
    fd  = {f["feature_id"]: f for f in inst.get("features", [])}
    q = inst.get("quality", {}); src = inst.get("source", {})
    view = src.get("camera_view","?")
    h = [CSS, '<div class="doh">']

    # 상단 요약 칩
    conf = int(round(q.get("overall_confidence",0)*100))
    vm = "✅ 적합" if q.get("view_match") else "⚠️ 부적합"
    h.append('<div class="meta">')
    for lab, key in [("촬영", "정면(FO)" if view=="FO" else ("측면(DTL)" if view=="DTL" else view)),
                     ("종합 신뢰도", f"{conf}%"), ("각도", vm), ("프레임", src.get("duration_frames","?"))]:
        h.append(f'<span class="chip">{lab} <b>{key}</b></span>')
    h.append("</div>")

    # ① P구간 이벤트 (P1~P10)
    h.append('<h2>① 스윙 P구간 (자동검출)</h2><div class="evrow">')
    for p in PORD:
        lb = PLBL[p]
        if p in ev:
            itp = " itp" if evm.get(p)=="interpolated" else ""
            tag = " ·보간" if evm.get(p)=="interpolated" else ""
            h.append(f'<span class="ev{itp}"><b>{p}</b> {lb} <span class="fr">#{ev[p]}{tag}</span></span>')
        else:
            h.append(f'<span class="ev miss"><b>{p}</b> {lb} <span class="fr">미검출</span></span>')
    h.append("</div>")
    h.append('<div class="note">P2·P6·P8(샤프트 평행)은 클럽 검출이 있어야 잡혀요 — 현재 포즈 엔진만으론 미검출(2차 클럽엔진 몫). 나머지는 3D 손높이·리드팔 수평으로 자동검출.</div>')

    # ② 회전 (3D 정밀)
    h.append('<h2>② 회전 (3D 정밀 · NLF)</h2><div class="mgrid">')
    h.append('<div class="mcard"><h4>백스윙탑 (P4)</h4>')
    h.append(_mrow("상체(어깨)", fd.get("VF015")))
    h.append(_mrow("하체(골반)", fd.get("VF018")))
    h.append(_mrow("X-Factor", fd.get("VF020"), "xf"))
    h.append('</div>')
    h.append('<div class="mcard"><h4>임팩트 (P7)</h4>')
    h.append(_mrow("힙 클리어", fd.get("VF075")))
    h.append('</div></div>')
    h.append('<div class="note">어드레스(P1)=0° 기준 누적 회전량. X-Factor = |상체−하체|(몸통 분리각). 브라우저 근사(-19°)와 달리 단일영상 3D 복원값.</div>')

    # ③ 나머지 지표 (자세·팔·무릎·스웨이·템포)
    byg = {g: [] for g in GROUPS}
    for f in inst.get("features", []):
        g = META.get(f["feature_id"], (None,"기타"))[1]
        if g in byg: byg[g].append(f)
    started = False
    for g in GROUPS:
        fs = byg.get(g)
        if not fs: continue
        if not started:
            h.append('<h2>③ 자세 · 팔 · 무릎 · 스웨이 · 템포</h2>'); started = True
        h.append(f'<div style="margin-top:6px;font-size:13px;font-weight:700;color:#e8edf2">{g}</div>')
        h.append('<div class="grid" style="margin-top:8px;margin-bottom:6px">')
        h.append("".join(_fc(f) for f in fs)); h.append("</div>")

    warns = [w for w in q.get("warnings", []) if not w.startswith("joints=") and not w.startswith("object_engine")]
    if warns:
        h.append(f'<div class="foot">참고: {" · ".join(warns)}</div>')
    h.append("</div>")
    return "".join(h)

import json, os
def analyze_fn(video, view, hand):
    hide = gr.DownloadButton(visible=False)
    if not video: return "<p style='color:#8b97a5'>영상을 올려주세요.</p>", hide
    J, fps = video_to_joints(video)
    if J.ndim != 3 or J.shape[0] < 5:
        return "<p style='color:#ff7eb6'>사람/프레임을 충분히 못 잡았어요. 전신이 크게 나오는 영상으로.</p>", hide
    inst = build_v1(J, skeleton="smpl", view=view, hand=hand, fps=fps, video_id="swing")
    # analyzer2(브라우저 프론트)에 불러올 수 있게 doh.vision.v1 JSON을 파일로 저장 → DownloadButton으로 제공
    path = os.path.abspath("doh_vision_v1.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(inst, f, ensure_ascii=False, indent=2)
    return render(inst), gr.DownloadButton(value=path, visible=True)

# analyzer2 v21 톤에 맞춘 다크 테마
THEME = gr.themes.Base(primary_hue="blue", neutral_hue="slate").set(
    body_background_fill="#0f1216", body_text_color="#e8edf2",
    block_background_fill="#1a1f27", block_border_color="#262d38",
    button_primary_background_fill="#4ea1ff", button_primary_text_color="#04121f",
    input_background_fill="#222c39",
)
with gr.Blocks(title="DOH 스윙분석", theme=THEME) as demo:
    gr.Markdown("## 🏌️ DOH Vision — 스윙분석 (3D 정밀 · NLF)\n"
                "영상 올리고 **[분석하기]** → P1~P10 · 회전 · 척추각 · 자세 · 팔 · 무릎 · 템포.  "
                "*(측면=척추/플레인 · 정면=스웨이/좌우틸트)*")
    with gr.Row():
        vid = gr.Video(label="스윙 영상 (mp4/mov)", sources=["upload"])
        with gr.Column():
            view = gr.Radio(["FO", "DTL"], value="FO", label="촬영 각도  (FO=정면 / DTL=측면)")
            hand = gr.Radio(["right", "left"], value="right", label="주손")
            btn = gr.Button("분석하기", variant="primary", size="lg")
    out = gr.HTML()
    dl = gr.DownloadButton("⬇ 결과 JSON 내려받기 (doh.vision.v1 → analyzer2에 불러오기)",
                           variant="secondary", visible=False)
    btn.click(analyze_fn, [vid, view, hand], [out, dl])

demo.launch(debug=False)